# 01 — Data quality and exploratory analysis

**Purpose:** establish that the data can carry the KPIs before any KPI is reported.

The checks here are the notebook counterpart of [`sql/01_data_validation.sql`](../sql/01_data_validation.sql).
Anything this notebook finds and *handles* is written into
[`sql/00_analytical_tables.sql`](../sql/00_analytical_tables.sql), so the fix lives in the
model rather than in a one-off notebook cell.

**Run order**

```bash
python scripts/build_db.py       # CSVs  -> data/olist.db
python scripts/run_analysis.py   # SQL   -> outputs/tables, outputs/figures
```

**Headline conclusion:** the source is unusually clean. Referential integrity is
intact, prices are all positive, no delivery precedes its purchase, and payments
reconcile to item totals within 0.02%. Four things *do* need handling, and each
is addressed in the model: a non-unique `review_id`, 775 item-less orders,
8 delivered orders without a delivery date, and sparse months at both ends of
the window.

In [1]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

DB = ROOT / "data" / "olist.db"
assert DB.exists(), "run `python scripts/build_db.py` first, then `python scripts/run_analysis.py`"
conn = sqlite3.connect(DB)
q = lambda sql: pd.read_sql_query(sql, conn)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("connected to", DB.relative_to(ROOT))

connected to data/olist.db


## 1. Shape and grain

The first question for any multi-table source: **what is one row of each table?**
Getting this wrong is how revenue gets double-counted.

In [2]:
q("""
SELECT 'orders.order_id' AS key_name, COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS distinct_ FROM orders
UNION ALL SELECT 'customers.customer_id', COUNT(*), COUNT(DISTINCT customer_id) FROM customers
UNION ALL SELECT 'customers.customer_unique_id', COUNT(*), COUNT(DISTINCT customer_unique_id) FROM customers
UNION ALL SELECT 'order_items.order_id', COUNT(*), COUNT(DISTINCT order_id) FROM order_items
UNION ALL SELECT 'order_payments.order_id', COUNT(*), COUNT(DISTINCT order_id) FROM order_payments
UNION ALL SELECT 'order_reviews.review_id', COUNT(*), COUNT(DISTINCT review_id) FROM order_reviews
UNION ALL SELECT 'order_reviews.order_id', COUNT(*), COUNT(DISTINCT order_id) FROM order_reviews
UNION ALL SELECT 'products.product_id', COUNT(*), COUNT(DISTINCT product_id) FROM products
UNION ALL SELECT 'sellers.seller_id', COUNT(*), COUNT(DISTINCT seller_id) FROM sellers
""")

,key_name,rows_,distinct_
0,orders.order_id,99441,99441
1,customers.customer_id,99441,99441
2,customers.customer_unique_id,99441,96096
3,order_items.order_id,112650,98666
4,order_payments.order_id,103886,99440
5,order_reviews.review_id,99224,98410
6,order_reviews.order_id,99224,98673
7,products.product_id,32951,32951
8,sellers.seller_id,3095,3095


Three results drive the whole model design:

1. **`customers` has 99,441 rows but only 96,096 distinct `customer_unique_id`.**
   `customer_id` is regenerated for every order — it is an *order* key wearing a
   customer's name. Keying retention on it reports a 0% repeat rate, and nothing
   about the query looks wrong. Every customer measure in this project uses
   `customer_unique_id`.

2. **`review_id` is not unique** (789 ids are reused) and neither is
   `order_reviews.order_id`. Joining reviews straight onto orders multiplies
   those orders. The model collapses reviews to order grain first
   (`order_review`).

3. **`order_items.order_id` repeats** — that is correct and expected (an order
   can hold several lines), but it means `orders ⋈ order_items` fans out, so
   money must be aggregated to order grain *before* any other join.

## 2. Referential integrity

In [3]:
q("""
SELECT 'order_items -> orders' AS relationship, COUNT(*) AS orphans
  FROM order_items i LEFT JOIN orders o USING(order_id) WHERE o.order_id IS NULL
UNION ALL SELECT 'orders -> customers', COUNT(*)
  FROM orders o LEFT JOIN customers c USING(customer_id) WHERE c.customer_id IS NULL
UNION ALL SELECT 'order_items -> products', COUNT(*)
  FROM order_items i LEFT JOIN products p USING(product_id) WHERE p.product_id IS NULL
UNION ALL SELECT 'order_items -> sellers', COUNT(*)
  FROM order_items i LEFT JOIN sellers s USING(seller_id) WHERE s.seller_id IS NULL
""")

,relationship,orphans
0,order_items -> orders,0
1,orders -> customers,0
2,order_items -> products,0
3,order_items -> sellers,0


Zero orphans on every foreign key. Now the reverse direction — rows with *no* children:

In [4]:
q("""
SELECT o.order_status, COUNT(*) AS orders_without_items
  FROM orders o
  LEFT JOIN (SELECT DISTINCT order_id FROM order_items) i USING(order_id)
 WHERE i.order_id IS NULL
 GROUP BY o.order_status ORDER BY orders_without_items DESC
""")

,order_status,orders_without_items
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


775 orders have no line items, and 767 of those are `unavailable` or
`canceled` — orders that never shipped. They carry no money, so the delivered-only
eligibility rule removes them from GMV without any special-casing.

## 3. Order status and the eligibility rule

Every KPI needs an explicit answer to "which orders count?" 

In [5]:
status = q("""
SELECT order_status, COUNT(*) AS orders,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM orders), 2) AS pct,
       SUM(order_delivered_customer_date IS NULL) AS missing_delivery_date
  FROM orders GROUP BY order_status ORDER BY orders DESC
""")
status

,order_status,orders,pct,missing_delivery_date
0,delivered,96478,97.02,8
1,shipped,1107,1.11,1107
2,canceled,625,0.63,619
3,unavailable,609,0.61,609
4,invoiced,314,0.32,314
5,processing,301,0.30,301
6,created,5,0.01,5
7,approved,2,0.00,2


`delivered` is 97.0% of all orders and is the only status with complete
money *and* delivery timestamps. Every other status is missing the delivery date
by definition — you cannot compute an on-time rate for an order that has not
arrived. **The reporting basis is `order_status = 'delivered'`.**

That choice has to be defended rather than assumed, so here is what the
alternatives would produce:

In [6]:
q("""
SELECT 'delivered (chosen basis)' AS basis, COUNT(*) AS orders, ROUND(SUM(gmv),2) AS gmv,
       ROUND(SUM(gmv)/COUNT(*),2) AS aov FROM order_base WHERE order_status='delivered'
UNION ALL SELECT 'not canceled/unavailable', COUNT(*), ROUND(SUM(gmv),2), ROUND(SUM(gmv)/COUNT(*),2)
  FROM order_base WHERE order_status NOT IN ('canceled','unavailable') AND item_count>0
UNION ALL SELECT 'all statuses', COUNT(*), ROUND(SUM(gmv),2), ROUND(SUM(gmv)/COUNT(*),2)
  FROM order_base WHERE item_count>0
""")

,basis,orders,gmv,aov
0,delivered (chosen basis),96478,13221498.11,137.04
1,not canceled/unavailable,98199,13494400.74,137.42
2,all statuses,98666,13591643.70,137.75


The widest basis adds 2.8% to GMV and 71 centavos to AOV. **No conclusion
in this project depends on the choice** — worth knowing, and worth stating, before
anyone asks.

## 4. Missing values

Which columns are actually missing, and does the gap matter?

In [7]:
q("""
SELECT COUNT(*) AS orders,
       SUM(order_approved_at IS NULL)             AS approved_null,
       SUM(order_delivered_carrier_date IS NULL)  AS carrier_null,
       SUM(order_delivered_customer_date IS NULL) AS delivered_null,
       SUM(order_estimated_delivery_date IS NULL) AS estimated_null
  FROM orders
""")

,orders,approved_null,carrier_null,delivered_null,estimated_null
0,99441,160,1783,2965,0


In [8]:
q("""
SELECT COUNT(*) AS delivered_orders,
       SUM(order_delivered_customer_date IS NULL) AS missing_delivery_date
  FROM orders WHERE order_status='delivered'
""")

,delivered_orders,missing_delivery_date
0,96478,8


**8 delivered orders have no delivery date.** They are dropped from
delivery KPIs rather than counted as late — `order_base` sets their
`is_on_time` to `NULL`, so they fall out of both numerator and denominator.
Scoring them as late would be a small but systematic overstatement of the
late rate, and the kind of default that is invisible once it is baked in.

## 5. Value ranges and date logic

A cheap sweep for impossible values.

In [9]:
q("""
SELECT 'item price' AS measure, MIN(price) AS min_, MAX(price) AS max_,
       SUM(price <= 0) AS non_positive FROM order_items
UNION ALL SELECT 'freight', MIN(freight_value), MAX(freight_value), SUM(freight_value < 0) FROM order_items
UNION ALL SELECT 'review score', MIN(review_score), MAX(review_score),
       SUM(review_score NOT BETWEEN 1 AND 5) FROM order_reviews
""")

,measure,min_,max_,non_positive
0,item price,0.85,6735.00,0
1,freight,0.00,409.68,0
2,review score,1.00,5.00,0


In [10]:
q("""
SELECT 'delivered before purchase' AS check_, COUNT(*) AS n
  FROM orders WHERE order_delivered_customer_date < order_purchase_timestamp
UNION ALL SELECT 'approved before purchase', COUNT(*)
  FROM orders WHERE order_approved_at < order_purchase_timestamp
UNION ALL SELECT 'delivered after estimate by >60d', COUNT(*)
  FROM orders WHERE julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) > 60
""")

,check_,n
0,delivered before purchase,0
1,approved before purchase,0
2,delivered after estimate by >60d,84


All clean. Prices run R$0.85–R$6,735 with none at or below zero, review
scores are all in 1–5, and no delivery precedes its own purchase. The handful of
extreme late deliveries (>60 days past estimate) are real tail events, not data
errors — they survive into the analysis.

## 6. The independent check: do payments reconcile to items?

Every check so far tests one table against itself. This one tests two
independently-recorded money columns against each other — the strongest
available evidence that the join grain is right.

In [11]:
q("""
WITH it AS (SELECT order_id, SUM(price) AS p, SUM(freight_value) AS f FROM order_items GROUP BY order_id),
     pm AS (SELECT order_id, SUM(payment_value) AS pv FROM order_payments GROUP BY order_id)
SELECT ROUND(SUM(it.p),2) AS items_price, ROUND(SUM(it.f),2) AS freight,
       ROUND(SUM(pm.pv),2) AS payments,
       ROUND(SUM(pm.pv)-SUM(it.p)-SUM(it.f),2) AS gap,
       ROUND(100.0*(SUM(pm.pv)-SUM(it.p)-SUM(it.f))/SUM(pm.pv),4) AS gap_pct
  FROM it JOIN pm USING(order_id)
""")

,items_price,freight,payments,gap,gap_pct
0,13591508.73,2251901.05,15846280.17,2870.39,0.0181


**R$2,870 on R$15.8M — a 0.02% gap.** Payments and item lines are
recorded separately in the source, so this near-match is strong evidence that
the roll-up grain is correct. A fan-out bug here would show up as a gap of tens
of percent, not hundredths.

(The residual is rounding plus a handful of vouchers and partial payments. It is
far too small to move any KPI.)

## 7. Period coverage — which months can be trended?

In [12]:
cov = q("""
SELECT order_month, COUNT(*) AS orders FROM order_base GROUP BY order_month ORDER BY order_month
""")
cov["status"] = cov.orders.apply(lambda n: "SPARSE — excluded" if n < 500 else "in trend window")
cov

,order_month,orders,status
0,2016-09,4,SPARSE — excluded
1,2016-10,324,SPARSE — excluded
2,2016-12,1,SPARSE — excluded
3,2017-01,800,in trend window
4,2017-02,1780,in trend window
5,2017-03,2682,in trend window
6,2017-04,2404,in trend window
7,2017-05,3700,in trend window
8,2017-06,3245,in trend window
9,2017-07,4026,in trend window


2016-09, 2016-10 and 2016-12 are a pilot-scale trickle (4, 324 and 1 orders).
2018-09 and 2018-10 are truncated mid-collection (16 and 4 orders).

Charting those months produces a dramatic-looking collapse at the right edge
that is pure collection artefact. **Every month-over-month figure in this
project is restricted to 2017-01 … 2018-08**; all-time totals use the full
dataset and say so.

## 8. Category labels

In [13]:
q("""
SELECT SUM(product_category_name IS NULL) AS products_with_no_category,
       COUNT(*) AS products FROM products
""")

,products_with_no_category,products
0,610,32951


In [14]:
q("""
SELECT p.product_category_name AS untranslated_category, COUNT(*) AS products
  FROM products p LEFT JOIN product_category_translation t USING(product_category_name)
 WHERE t.product_category_name_english IS NULL AND p.product_category_name IS NOT NULL
 GROUP BY 1 ORDER BY 2 DESC
""")

,untranslated_category,products
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


610 products carry no category at all, and two live categories
(`portateis_cozinha_e_preparadores_de_alimentos`, `pc_gamer`) are simply absent
from the vendor's translation file. `product_dim` patches the two translations
by hand and maps the rest to `'unknown'` rather than `NULL`, so those
R$170k of sales stay visible in category reporting instead of silently
disappearing from a `GROUP BY`.

## 9. First look at the shape of demand

In [15]:
trend = pd.read_csv(ROOT / "outputs" / "tables" / "monthly_trend.csv")
trend

,order_month,orders,gmv,aov,active_customers,gmv_mom_pct
0,2017-01,750,111798.36,149.06,718,NaN
1,2017-02,1653,234223.40,141.70,1630,109.51
2,2017-03,2546,359198.85,141.08,2508,53.36
3,2017-04,2303,340669.68,147.92,2274,-5.16
4,2017-05,3546,489338.25,138.00,3479,43.64
5,2017-06,3135,421923.37,134.58,3076,-13.78
6,2017-07,3872,481604.52,124.38,3802,14.15
7,2017-08,4193,554699.70,132.29,4114,15.18
8,2017-09,4150,607399.67,146.36,4083,9.50
9,2017-10,4478,648247.65,144.76,4417,6.73


In [16]:
q("""
SELECT order_status, COUNT(*) AS orders, ROUND(AVG(gmv),2) AS avg_gmv,
       ROUND(AVG(item_count),2) AS avg_items
  FROM order_base GROUP BY order_status ORDER BY orders DESC
""")

,order_status,orders,avg_gmv,avg_items
0,delivered,96478,137.04,1.14
1,shipped,1107,136.16,1.07
2,canceled,625,152.38,0.87
3,unavailable,609,3.30,0.01
4,invoiced,314,195.94,1.14
5,processing,301,200.79,1.19
6,created,5,0.00,0.00
7,approved,2,104.80,1.50


In [17]:
q("""
SELECT item_count AS items_in_order, COUNT(*) AS orders,
       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
  FROM order_base WHERE is_eligible=1 GROUP BY item_count ORDER BY item_count LIMIT 8
""")

,items_in_order,orders,pct
0,1,86843,90.01
1,2,7392,7.66
2,3,1306,1.35
3,4,495,0.51
4,5,193,0.20
5,6,191,0.20
6,7,22,0.02
7,8,8,0.01


**Baskets are tiny.** 90% of delivered orders contain exactly one item,
and the mean is 1.14. This single fact shapes the recommendations: there is far
more headroom in attach-rate and bundling than in discounting, because almost
nobody is buying two things at once today.

## 10. Summary of handling decisions

| Finding | Handling | Where |
| --- | --- | --- |
| `customer_id` is per-order, not per-person | all customer metrics key on `customer_unique_id` | `customer_base` |
| `review_id` and `order_reviews.order_id` non-unique | collapse to order grain by mean score | `order_review` |
| `order_items` fans out against `orders` | aggregate money to order grain before joining | `order_base` |
| 775 orders with no items | excluded by the delivered-only rule | `is_eligible` |
| 8 delivered orders with no delivery date | delivery measures set to `NULL`, not 0 | `order_base` |
| 610 products with no category, 2 untranslated | mapped to `'unknown'`; 2 translations patched | `product_dim` |
| sparse months at both ends | trends limited to 2017-01 … 2018-08 | all trend queries |

Continue to [`02_customer_analysis.ipynb`](02_customer_analysis.ipynb).

In [18]:
conn.close()